In [1]:
import os
import ast
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchaudio
import torchaudio.functional as AF

import soundfile as sf
import librosa
import timm

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split, StratifiedGroupKFold


torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')



# Select device and Hyperparameters
n_epochs = 20      # Full dataset passes
batch_size = 32    # Samples per gradient step
gamma = -0.5       # Sampler balancing exponent
p_noise = 0.50     # Background noise augmentation probability
p_mix = 0.55       # MixUp augmentation probability
p_filter = 0.25    # Random filtering augmentation probability
lr = 1e-4          # Learning rate
wd = 1e-4          # Weight decay regularization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"""
Training configuration
----------------------
Device              : {device}
Epochs              : {n_epochs}
Batch size          : {batch_size}
Gamma               : {gamma}
Noise probability   : {p_noise}
MixUp probability   : {p_mix}
Filter probability  : {p_filter}
Learning rate       : {lr}
Weight decay        : {wd}
""")




# Select labesl
SELECTED_LABELS = ["whtdov", "undtin1", "compau", "trsowl"]


# Load tables and select
df_taxonomy = pd.read_csv("./data/taxonomy_mini.csv")
df_train = pd.read_csv("./data/train_mini.csv")
df_soundscapes = pd.read_csv("./data/train_soundscapes_labels_mini.csv") 

df_taxonomy = df_taxonomy[df_taxonomy["primary_label"].isin(SELECTED_LABELS)]
df_train = df_train[df_train["primary_label"].isin(SELECTED_LABELS)]

# Soundscapes
df_soundscapes["primary_label"] = (
    df_soundscapes["primary_label"]
    .astype(str)
    .apply(
        lambda x: ";".join(
            [
                label
                for label in x.split(";")
                if label in SELECTED_LABELS
            ]
        )
    )
)
df_soundscapes = df_soundscapes[
    df_soundscapes["primary_label"].apply(len) > 0
].reset_index(drop=True)

#  SAVE NEW TABLES
df_taxonomy.to_csv("./taxonomy_mini_ready.csv", index = False)
df_train.to_csv("./train_mini_ready.csv", index = False)
df_soundscapes.to_csv("./train_soundscapes_labels_mini_ready.csv", index = False)

# Load tables
df_taxonomy = pd.read_csv("./taxonomy_mini_ready.csv")
df_train = pd.read_csv("./train_mini_ready.csv")
df_soundscapes = pd.read_csv("./train_soundscapes_labels_mini_ready.csv") 


# Dictionary
path_csv = os.path.expanduser("./taxonomy_mini_ready.csv")
df = pd.read_csv(path_csv)
primary_label = df["primary_label"].unique()

# Create label to target dict
n_classes = len(primary_label)
DICTIONARY_LABEL2TARGET = {}
for i, label in enumerate(primary_label):
    vec = np.zeros(n_classes) 
    vec[i] = 1
    DICTIONARY_LABEL2TARGET[label] = vec

# Create a dictionary label to sample weight (sampled by train audio only)
df_audio = pd.read_csv("./train_mini_ready.csv")
temp_dict = df_audio["primary_label"].value_counts().to_dict()
total_count = len(df_audio)

DICTIONARY_LABEL2WEIGHT = {}
for label, count in temp_dict.items():
    w = (count / total_count)
    DICTIONARY_LABEL2WEIGHT[label] = w



class UtilFunctions():
    '''
    Utility class for handling label preprocessing in multi-label
    audio classification tasks.

    Includes:
    - Multi-hot encoding generation
    - Label aggregation for soundscapes
    - Label smoothing for regularization
    '''

    def __init__(self):
        # Initialize dictionaries and number of classes
        self.label2target = DICTIONARY_LABEL2TARGET
        self.label2weight = DICTIONARY_LABEL2WEIGHT
        self.n_classes = len(self.label2target)

    def multi_hot_audios(self, primary_label, secondary_labels):
        # Start from primary label one-hot vector (copy to avoid mutation)
        vec_1 = self.label2target[primary_label].copy()

        # If no secondary labels, return primary vector
        if secondary_labels == "[]":
            return vec_1
        else:
            # Parse string representation of list into Python list
            s_labels = ast.literal_eval(secondary_labels)

            # Combine vectors via element-wise max (multi-hot encoding)
            for label in s_labels:
                if label in self.label2target:
                    vec_2 = self.label2target[label]
                    vec_1 = np.maximum(vec_1, vec_2)
            return vec_1
        
    def multi_hot_soundscapes(self, primary_label):
        # Split multiple labels separated by ';'
        p_labels = primary_label.split(";")

        # Initialize empty multi-hot vector
        vec_1 = np.zeros(self.n_classes)

        # Aggregate all label vectors into a single multi-hot vector
        for label in p_labels:
            vec_2 = self.label2target[label]
            vec_1 = np.maximum(vec_1, vec_2)
        return vec_1
    
    # LABEL SMOOTHING (For knowledge distilation)
    def label_smooth(self, y_true, mu = 0.05):
        y_true_smooth = y_true * (1 - mu)  +  (mu * torch.sum(y_true, dim=1, keepdim=True) / self.n_classes) 
        return y_true_smooth


class LossCombined(nn.Module):
    """Multilabel loss: combination of BCE and Focal Loss."""
    
    def __init__(self, beta = 0.3, rho = 2):
        super().__init__()
        self.beta = beta  # If beta=0.3: final_loss = 0.3 * BCE + 0.7 * Focal
        self.rho = rho    
        self.bce = nn.BCELoss(reduction = "none")   # reduction="none" -> do NOT average yet, we want per-element loss
        self.epsilon = 1e-7

    def loss_focal(self, y_pred, y_true):
        val_1 = y_true * ( (1 - y_pred) ** self.rho ) * torch.log(y_pred)
        val_0 = (1 - y_true) * ( y_pred ** self.rho ) * torch.log(1 - y_pred)
        sum = -val_1 - val_0
        return sum

    def forward(self, y_logits, y_true):
        
        # Transform logit to prob
        y_pred = torch.sigmoid(y_logits)
        y_pred = torch.clamp(y_pred, self.epsilon, 1-self.epsilon)

        # calculate BCE elementwise
        bce_loss = self.bce(y_pred, y_true)

        # calculate focal elementwise
        focal_loss = self.loss_focal(y_pred, y_true)

        # Combine both functions
        loss = self.beta * bce_loss + ((1-self.beta) * focal_loss)

        return loss.mean()


# path to main dataframe 
path_df = "./train_mini_ready.csv" 

# Load table with data info 
df = pd.read_csv(path_df) 
print(f"The dim of train table before transforming data is {df.shape}")
print(f"With columns: {list(df.columns)}")

# PREPROCESS DATA
# Delete specific columns
columns_deleted = ['common_name', "class_name", 'scientific_name', 
                   'inat_taxon_id', 'license', 'url', "latitude","longitude", "type"] 
df = df.drop(columns=columns_deleted).copy()

# Transform "collection" to one-hot for future partition and validation, and delete collection column
mask = (df["collection"] == "iNat")
df["iNat"] =  np.array(mask).astype(int)
df["XC"] = np.array(~mask).astype(int)
df = df.drop(columns="collection").copy()

# Load taxonomy table and count all classes (234)
path_taxonomy = "./taxonomy_mini_ready.csv"
df_taxonomy =  pd.read_csv(path_taxonomy)
primary_label = df_taxonomy["primary_label"].unique()
print(f"The total number of classes is {len(primary_label)}")

# Add new column "sample_weight" using precomputed dictionary
df["sample_weight"] = df["primary_label"].map(DICTIONARY_LABEL2WEIGHT)

# Print Final Dataframe to use
print(f"\nThe dim of table after transforming data is {df.shape}")
print(f"With columns: {list(df.columns)}")

# S P L I T 
# fill Missing values of author with "unknown"
df["author"] = df["author"].fillna("unknown").astype(str)

# Split data acording to y = "primary_label", and grouped by "autor"
folds = 5
cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=666)
df["fold"] = -1
for fold, (_, val_idx) in enumerate(cv.split(df, y=df["primary_label"], groups=df["author"])):
    df.loc[val_idx, "fold"] = fold

# Look how many vals per fold
print("Data per fold:")
print(df["fold"].value_counts().sort_index())

# Save dfs
path_to_save = "./train_folds_mini_ready.csv"
df.to_csv(path_to_save, index=False)
print(f"\nThe train dataframes were created and saved succesfully at {path_to_save}")
print(f"Number of features: {len(df.columns)-1}")


class BirdAudioDataset(Dataset):
    '''
    Inputs:
        - df: transformed df of "train.csv" containing data, target and k-Folds index
        - segment_seconds: final duration of audio in seconds after cutting it

    Outputs:
        - chunk: Tensor [audio sample]
        - target: Tensor [num_classes]
        - (optional) metadata: Tensor [6]

    Description:
    Loads audio, extracts fixed-length segment, applies augmentation,
    returns waveform and multi-label target.
    '''

    def __init__(self, df, segment_seconds = 5, p_noise = 0.5, p_filter = 0.25, p_mix = 0.66):
        self.df = df.copy()
        self.utils = UtilFunctions()
        self.path_audios = os.path.expanduser("~/1_machine_learning_projects/BIRDS/data/train_audio")
        self.path_soundscapes = os.path.expanduser("~/1_machine_learning_projects/BIRDS/data/train_soundscapes")
        self.segment_seconds = segment_seconds  # to cut audio in segments of 5s
        self.soundscape_files = os.listdir(self.path_soundscapes)
        self.audio_files = os.listdir(self.path_audios)
        self.num_samples = len(self.df)
        self.p_noise = p_noise
        self.p_filter = p_filter
        self.p_mix = p_mix

    # Function to cut audio into "segment_seconds"
    def cut_audio(self, waveform, sample_rate=32000):
        # Verify if audio is larger than segment_seconds (5s)
        length_waveform = len(waveform)
        segment_samples = int(self.segment_seconds * sample_rate)
        
        if length_waveform <= segment_samples:
            # Add missing seconds as 0-values
            missing_segment = np.zeros( segment_samples - length_waveform, dtype = waveform.dtype)
            waveform = np.concatenate([waveform, missing_segment])      
        else:
            # Cut a 5s chunk randomly from waveform
            start = np.random.randint(0, (length_waveform - segment_samples)+1)
            end = start + segment_samples
            waveform = waveform[start:end]
        return waveform    

    # Function to add noise from train soundscapes
    def random_add_noise(self, waveform):
        path_soundscapes = self.path_soundscapes
        if random.random() < self.p_noise:
            # Create path to noise
            random_filename = random.choice(self.soundscape_files)
            path_filename = os.path.join(path_soundscapes, random_filename)

            # Load, cut and normalize noise
            noise, sample_rate = sf.read(path_filename)
            if noise.ndim == 2:
                noise = noise.mean(axis=1)
            noise = noise / (np.max(np.abs(noise)) + 1e-8)
            noise_chunk = self.cut_audio(noise, sample_rate)

            # add noise to waveform
            alpha = random.uniform(0.1, 0.3)
            waveform = waveform + (alpha * noise_chunk)
        return waveform

    # Function to Random filtering
    def random_filter(self, waveform, sample_rate=32000):
        if random.random() > self.p_filter:
            return waveform
        waveform = torch.tensor(waveform, dtype=torch.float32)
        waveform = AF.equalizer_biquad(waveform.unsqueeze(0), sample_rate=sample_rate, 
                        center_freq=random.uniform(500, 8000), gain=random.uniform(-6, 6), Q=random.uniform(0.5, 1.5)).squeeze(0)
        return waveform.numpy()
        

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Get rows
        row = self.df.iloc[idx]
        
        # Load audio filename and create path to file 
        filename = row["filename"]
        path_filename = os.path.join(self.path_audios, filename)

        # Extract waveform and 5s chunk
        waveform_1, _ = sf.read(path_filename)
        if waveform_1.ndim == 2:
            waveform_1 = waveform_1.mean(axis=1)
        chunk_1 = self.cut_audio(waveform_1)

        # Background mixing and Filtering
        chunk_1 = self.random_add_noise(chunk_1)
        chunk_1 = self.random_filter(chunk_1)

        # Noramlize and transform to tensor
        chunk_1 = chunk_1 / (np.max(np.abs(chunk_1)) + 1e-8)
        chunk_1 = torch.tensor(chunk_1, dtype = torch.float32)

        # Get target
        target_1 = torch.tensor(self.utils.multi_hot_audios(row["primary_label"], row["secondary_labels"]), dtype=torch.float32)
 
        # Return chunk and target or randomly apply MixUP(add a second waveform to first one randomly)
        if random.random() > self.p_mix:
            return chunk_1, target_1
        
        # MixUp
        idx_2 = random.randint(0, self.num_samples - 1)  
        if idx_2 == idx:
            idx_2 = idx_2 - 1
        row_2 = self.df.iloc[idx_2]
        waveform_2, _ = sf.read(os.path.join(self.path_audios, row_2["filename"]))
        if waveform_2.ndim == 2:
            waveform_2 = waveform_2.mean(axis=1)
        
        # Cut second waveform, randomly apply filter and noise, normalize and convert to torch
        chunk_2 = self.cut_audio(waveform_2)
        chunk_2 = self.random_add_noise(chunk_2)
        chunk_2 = self.random_filter(chunk_2)
        chunk_2 = chunk_2 / (np.max(np.abs(chunk_2)) + 1e-8)
        chunk_2 = torch.tensor(chunk_2, dtype = torch.float32)

        # Get second target
        target_2 = torch.tensor(self.utils.multi_hot_audios(row_2["primary_label"], row_2["secondary_labels"]),dtype=torch.float32)

        # Combine target and chunks
        target = torch.maximum(target_1, target_2)
        chunk = chunk_1 + chunk_2

        return chunk, target


class BirdEfficientNetV2S(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, dropout=0.25, n_fft=2048, hop_length=512, n_mels=128):

        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        
        # CNN backbone
        self.backbone = timm.create_model("tf_efficientnetv2_s.in21k_ft_in1k", 
                        pretrained=pretrained, in_chans=1, num_classes=0, global_pool="avg")

        # Audio to Log-Mel
        self.mel = torchaudio.transforms.MelSpectrogram(sample_rate=32000, 
                        n_fft=self.n_fft, hop_length=self.hop_length, n_mels=self.n_mels, power=2.0)
        self.db = torchaudio.transforms.AmplitudeToDB()

        # Classification head
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, num_classes)
        )

    def forward(self, x):
        """
        Args = x: input tensor of shape [B, 32kHz * 5s]

        Returns =  y: raw class scores of shape [B, num_classes]
        """
        # Transform waveform to log-mel
        x = self.mel(x)  # [B,128,T]
        x = self.db(x)  
        # Normalize
        x = (x - x.mean(dim=(1,2), keepdim=True)) / (x.std(dim=(1,2), keepdim=True) + 1e-6)
        # CNN
        x = x.unsqueeze(1)  # [B,1,128,T]
        x = self.backbone(x)  # [B,feat_dim]
        # Head
        y = self.head(x)  # [B,234]

        return y




# Load dataframe 
path = os.path.expanduser("./train_folds_mini_ready.csv")
df = pd.read_csv(path)
n_folds = df["fold"].nunique()


# K = 5 Fold
for fold in range(n_folds):

    print(f"\n===== Fold {fold + 1}/{n_folds} =====")
    # Split train / validation folds
    df_train = df[df["fold"] != fold].reset_index(drop=True)
    df_val = df[df["fold"] == fold].reset_index(drop=True)
    
    # Create datasets
    data_train = BirdAudioDataset(df=df_train, p_noise=p_noise, p_filter=p_filter, p_mix=p_mix)
    data_val = BirdAudioDataset(df=df_val, p_noise=0.0, p_filter=0.0, p_mix=0.0)

    # Create train dataloader using sampler w load weights
    sample_weights = torch.tensor(df_train["sample_weight"].values ** gamma, dtype=torch.double) # Loader weights
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    loader_train = DataLoader(data_train, batch_size=batch_size, sampler=sampler, 
        shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)

    # Create validation
    loader_val = DataLoader(data_val, batch_size=batch_size, shuffle=False, 
        num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)

    # Models
    model_1 = BirdEfficientNetV2S(dropout=0.25, n_fft=2048, hop_length=512, n_mels=128).to(device)
    model_2 = BirdEfficientNetV2S(dropout=0.25, n_fft=1024, hop_length=320, n_mels=128).to(device)

    # Loss + optimizer
    criterion_1 = LossCombined(beta = 0.3, rho = 2) # If beta=0.3: final_loss = 0.3 * BCE + 0.7 * Focal
    criterion_2 = LossCombined(beta = 0.5, rho = 2)
    optimizer_1 = torch.optim.AdamW(model_1.parameters(), lr=lr, weight_decay=wd)
    optimizer_2 = torch.optim.AdamW(model_2.parameters(), lr=lr, weight_decay=wd)

    # Learning rate scheduler
    scheduler_1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_1, T_max=n_epochs, eta_min=1e-6)
    scheduler_2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_2, T_max=n_epochs, eta_min=1e-6)

    # Smooth y_true
    smooth_y = UtilFunctions() 

    
    # =========================== TRAINING LOOP =================================
    for epoch in range(n_epochs):
    
        # Put models in train mode
        model_1.train()
        model_2.train()
    
        # Create progress bar
        pbar = tqdm(loader_train, desc=f"Epoch {epoch+1}/{n_epochs}")
    
        for batch_X, batch_y in pbar:
                
            # To cpu
            batch_X = batch_X.to(device, non_blocking=True)
            batch_y = batch_y.float().to(device, non_blocking=True)
                
            # Reinitialize gradients
            optimizer_1.zero_grad(set_to_none=True)
            optimizer_2.zero_grad(set_to_none=True)
    
            # Forward
            logits_1 = model_1(batch_X)
            logits_2 = model_2(batch_X)
    
            # Smooth target
            batch_y_s = smooth_y.label_smooth(batch_y)
    
            # Loss
            loss_1 = criterion_1(logits_1, batch_y_s)
            loss_2 = criterion_2(logits_2, batch_y_s)
    
            # Backprop
            loss_1.backward()
            loss_2.backward()
    
            optimizer_1.step()
            optimizer_2.step()
    
    
        # Scheduler
        scheduler_1.step()
        scheduler_2.step()

        # ======================= VALIDATION LOOP ===============================
        
        model_1.eval()
        model_2.eval()
        
        val_loss_1 = 0.0
        val_loss_2 = 0.0

        with torch.no_grad():

            for batch_X, batch_y in loader_val:

                # To gpu
                batch_X = batch_X.to(device, non_blocking=True)
                batch_y = batch_y.float().to(device, non_blocking=True)

                # Forward
                logits_1 = model_1(batch_X)
                logits_2 = model_2(batch_X)

                # Validation loss
                loss_1 = criterion_1(logits_1, batch_y)
                loss_2 = criterion_2(logits_2, batch_y)

                val_loss_1 += loss_1.item()
                val_loss_2 += loss_2.item()

        # Average validation losses
        val_loss_1 /= len(loader_val)
        val_loss_2 /= len(loader_val)


        # Print validation metrics
        print(
            f"[Fold {fold + 1}] "
            f"Epoch {epoch + 1}/{n_epochs} | "
            f"Val Loss 1: {val_loss_1:.4f} | "
            f"Val Loss 2: {val_loss_2:.4f}"
        )

        
# Save Model
torch.save(model_1.state_dict(), "output/model_1.pt")
torch.save(model_2.state_dict(), "output/model_2.pt")


print("Models saved at output/model.pt")


Training configuration
----------------------
Device              : cuda
Epochs              : 20
Batch size          : 32
Gamma               : -0.5
Noise probability   : 0.5
MixUp probability   : 0.55
Filter probability  : 0.25
Learning rate       : 0.0001
Weight decay        : 0.0001

The dim of train table before transforming data is (1679, 15)
With columns: ['primary_label', 'secondary_labels', 'type', 'latitude', 'longitude', 'scientific_name', 'common_name', 'class_name', 'inat_taxon_id', 'author', 'license', 'rating', 'url', 'filename', 'collection']
The total number of classes is 4

The dim of table after transforming data is (1679, 8)
With columns: ['primary_label', 'secondary_labels', 'author', 'rating', 'filename', 'iNat', 'XC', 'sample_weight']
Data per fold:
fold
0    309
1    318
2    349
3    406
4    297
Name: count, dtype: int64

The train dataframes were created and saved succesfully at ./train_folds_mini_ready.csv
Number of features: 8

===== Fold 1/5 =====


Epoch 1/20:   0%|                                                                                                        | 0/43 [00:47<?, ?it/s]


KeyboardInterrupt: 